# MD — alanine dipeptide (water, 300 K)

Data: `input_data/MD-water-300K/` (`topol.tpr`, `traj.trr`). See `input_data/README.md` if the trajectory files are missing.


In [ ]:
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import MDAnalysis as mda
import numpy as np

from pyfresean import FRESEAN, Align, Unwrap
from pyfresean.postprocess import (
    cluster_mode_indices,
    low_frequency_peaks,
    mode_overlap_matrix,
    mode_spectra,
    mode_time_correlation,
    plot_spectra,
    select_modes_in_frequency_range,
    sort_clusters_by_peak_frequency,
    weighted_similarity_matrix,
)

INPUT_DATA = Path("input_data")
INPUT_DATA.mkdir(parents=True, exist_ok=True)
OUTPUT_DATA = Path("output_data")
OUTPUT_DATA.mkdir(parents=True, exist_ok=True)


def download_if_needed(path, url):
    path = Path(path)
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        print(f"Downloading {path} ...")
        urllib.request.urlretrieve(url, path)


In [ ]:
from MDAnalysis.analysis.dihedrals import Dihedral
from MDAnalysis.analysis.rms import rmsd
from tqdm import tqdm


def longest_consecutive_indices(array1, lim1, array2, lim2, lim1alt=None, lim2alt=None):
    if lim1alt is None and lim2alt is None:
        mask = (array1 >= lim1[0]) & (array1 <= lim1[1]) & (array2 >= lim2[0]) & (array2 <= lim2[1])
    elif lim1alt is None:
        mask = (array1 >= lim1[0]) & (array1 <= lim1[1]) & (
            ((array2 >= lim2[0]) & (array2 <= lim2[1]))
            | ((array2 >= lim2alt[0]) & (array2 <= lim2alt[1]))
        )
    elif lim2alt is None:
        mask = (
            ((array1 >= lim1[0]) & (array1 <= lim1[1]))
            | ((array1 >= lim1alt[0]) & (array1 <= lim1alt[1]))
        ) & (array2 >= lim2[0]) & (array2 <= lim2[1])
    else:
        mask = (
            ((array1 >= lim1[0]) & (array1 <= lim1[1]))
            | ((array1 >= lim1alt[0]) & (array1 <= lim1alt[1]))
        ) & (
            ((array2 >= lim2[0]) & (array2 <= lim2[1]))
            | ((array2 >= lim2alt[0]) & (array2 <= lim2alt[1]))
        )

    max_len = 0
    max_start = -1
    current_len = 0
    current_start = -1
    for i, val in enumerate(mask):
        if val:
            if current_len == 0:
                current_start = i
            current_len += 1
            if current_len > max_len:
                max_len = current_len
                max_start = current_start
        else:
            current_len = 0
    if max_len == 0:
        return None
    return list(range(max_start, max_start + max_len))


topol = INPUT_DATA / "MD-water-300K" / "topol.tpr"
traj = INPUT_DATA / "MD-water-300K" / "traj.trr"
ref_path = OUTPUT_DATA / "MD-water-300K" / "conformation-1.xyz"

download_if_needed(topol, "https://www.dropbox.com/scl/fi/aydgkxfbtbeif7j7ptr5o/topol.tpr?rlkey=g4bztzt95dt402spumdxmz66n&dl=1")
download_if_needed(traj, "https://www.dropbox.com/scl/fi/m0dlzyeaijw48wj8gky18/traj.trr?rlkey=gggwkd2e9rfwv8or13qgo3ulq&dl=1")

u = mda.Universe(str(topol), str(traj))
sel = u.select_atoms("all")

phi = Dihedral([sel.residues[1].phi_selection()]).run()
psi = Dihedral([sel.residues[1].psi_selection()]).run()
traj_indices = longest_consecutive_indices(
    phi.results.angles.T[0], [-180, 0], psi.results.angles.T[0], [100, 180], lim2alt=[-180, -150]
)
print(f"segment: {len(traj_indices)} frames")

n_corr = 500
dt = 0.004
sigma = 10.0
n_constraints = 0.02718

u.trajectory.add_transformations(Unwrap(u.atoms))

rmsd_frames = traj_indices[::25]
rmsd_matrix = np.zeros((len(rmsd_frames), len(rmsd_frames)))
for i in tqdm(range(len(rmsd_frames))):
    u.trajectory[rmsd_frames[i]]
    a = sel.positions.copy()
    for j in range(i + 1, len(rmsd_frames)):
        u.trajectory[rmsd_frames[j]]
        b = sel.positions.copy()
        x = rmsd(a, b, center=True, superposition=True)
        rmsd_matrix[i, j] = x
        rmsd_matrix[j, i] = x

row_counts = np.sum(rmsd_matrix < 0.75, axis=1)
max_row_index = int(np.where(row_counts == np.max(row_counts))[0][-1])
ref_path.parent.mkdir(parents=True, exist_ok=True)
with mda.Writer(str(ref_path), sel.n_atoms) as w:
    u.trajectory[rmsd_frames[max_row_index]]
    w.write(sel)

u_ref = mda.Universe(str(ref_path))
u.trajectory.add_transformations(
    Align(sel, reference_positions=u_ref.atoms.positions, place_com_in_box=False),
)

analysis = FRESEAN(u, select="all", n_constraints=n_constraints, n_corr=n_corr, dt=dt, sigma=sigma)
analysis.run(frames=traj_indices)


In [ ]:
freqs = analysis.results.freqs
vdos = analysis.results.vdos_total
eigenvalues = analysis.results.eigenvalues
eigenvectors = analysis.results.eigenvectors
corr = analysis.results.corr_matrix
win_time = analysis.results.win_time
avg_temp = analysis.results.avg_temperature
n_corr = analysis.n_corr
dt = analysis.dt
n_modes = 4
peak_choice = 0

low_peaks = low_frequency_peaks(freqs, vdos, max_freq=200.0, extra=(11,))
print(f"T = {avg_temp:.1f} K")
print(f"peaks (cm-1): {freqs[low_peaks]}")


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
plot_spectra(
    freqs, vdos, ax=ax, xlim=(0, 200), ylim=(0, 0.6),
    labels="Total VDoS", vlines=freqs[low_peaks].tolist(),
)
plt.tight_layout()
plt.show()


In [ ]:
zero_modes = eigenvectors[0, :n_modes]
zero_vdos = mode_spectra(corr, zero_modes)
sum_zero = np.sum(zero_vdos, axis=0)

fig, ax = plt.subplots(figsize=(6, 4))
plot_spectra(
    freqs,
    [vdos, *zero_vdos, sum_zero],
    ax=ax,
    xlim=(0, 200),
    ylim=(0, 0.6),
    colors=["black", "red", "orange", "gold", "green", "gray"],
    labels=["Total VDoS", "Mode 1", "Mode 2", "Mode 3", "Mode 4", "Sum of modes"],
    linestyles=["-", "--", "--", "--", "--", "-"],
    title="0.0 cm$^{-1}$",
)
plt.tight_layout()
plt.show()


In [ ]:
peak_idx = low_peaks[peak_choice]
mode_vdos = mode_spectra(corr, eigenvectors[peak_idx, :n_modes])
sum_modes = np.sum(mode_vdos, axis=0)

fig, ax = plt.subplots(figsize=(6, 4))
plot_spectra(
    freqs,
    [vdos, *mode_vdos, sum_modes],
    ax=ax,
    xlim=(0, 200),
    ylim=(0, 0.6),
    colors=["black", "red", "orange", "gold", "green", "gray"],
    labels=["Total VDoS", "Mode 1", "Mode 2", "Mode 3", "Mode 4", "Sum of modes"],
    linestyles=["-", "--", "--", "--", "--", "-"],
    vlines=[freqs[peak_idx]],
    title=f"{freqs[peak_idx]:.1f} cm$^{-1}$",
)
plt.tight_layout()
plt.show()


In [ ]:
freq_indices = [0] + low_peaks.tolist()
fig, axs = plt.subplots(len(freq_indices), 1, figsize=(6, 1.2 * len(freq_indices)), sharex=True)
if len(freq_indices) == 1:
    axs = [axs]

for ax, freq_idx in zip(axs, freq_indices):
    vals = eigenvalues[freq_idx, :10]
    ax.plot(np.arange(1, 11), vals, "o-", color="black")
    ax.axhline(0, color="black", linestyle=":", linewidth=0.8)
    ax.set_ylabel("VDoS")
    ax.set_title(f"{freqs[freq_idx]:.1f} cm$^{-1}$")
    ax.set_xlim(0.5, 10.5)

axs[-1].set_xlabel("eigenvalue number")
plt.tight_layout()
plt.show()


In [ ]:
peak_idx = low_peaks[peak_choice]
times = np.arange(n_corr) * dt

fig, axs = plt.subplots(5, 5, figsize=(6, 6), sharex=True, sharey=True)
for i, axrow in enumerate(axs):
    for j, ax in enumerate(axrow):
        vcf, vcf_actual = mode_time_correlation(
            corr, eigenvectors[peak_idx, i], eigenvectors[peak_idx, j],
            n_corr, win_time, avg_temp,
        )
        ax.plot(times, vcf_actual, color="black")
        ax.plot(times, vcf, color="red")
        ax.set_xlim(0, 2)
        ax.set_ylim(-330, 330)
        ax.axhline(0, color="black", linestyle=":", linewidth=0.8)
        ax.grid(True)

axs[-1][2].set_xlabel("correlation time (ps)")
axs[2][0].set_ylabel(r"time correlation function / k$_{\mathrm{b}}$ (K)")
plt.suptitle(f"{freqs[peak_idx]:.1f} cm$^{-1}$")
for i, ax in enumerate(axs[0]):
    ax.set_title(f"Mode {i + 1}", fontsize=10)
plt.tight_layout()
plt.show()


In [ ]:
vec_sel, val_sel = select_modes_in_frequency_range(freqs, eigenvalues, eigenvectors)
cluster_matrix = weighted_similarity_matrix(
    val_sel, vec_sel, rigid_modes=eigenvectors[0, :6]
)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cluster_matrix, aspect="auto", cmap="inferno", vmin=0, vmax=1)
fig.colorbar(im, ax=ax, label="weighted mode similarity")
ax.set_xlabel("selected mode index")
ax.set_ylabel("selected mode index")
plt.tight_layout()
plt.show()


In [ ]:
cutoff = 0.3
cluster_idx = cluster_mode_indices(cluster_matrix, cutoff=cutoff)
cluster_modes = vec_sel[cluster_idx]
cluster_vdos = mode_spectra(corr, cluster_modes)
sort_idx, cluster_peak_idx = sort_clusters_by_peak_frequency(cluster_vdos)
cluster_modes = cluster_modes[sort_idx]
cluster_vdos = cluster_vdos[sort_idx]
cluster_peak_idx = cluster_peak_idx
print(f"Found {len(cluster_modes)} key vibrations (cutoff={cutoff})")


In [ ]:
n_plot = min(7, len(cluster_modes))
plot_n = min(25, len(freqs))
plot_freqs = freqs[:plot_n]
cluster_colors = ["red", "orange", "gold", "green", "cyan", "blue", "magenta"]

fig, ax = plt.subplots(figsize=(6, 4))
plot_spectra(
    plot_freqs,
    [vdos[:plot_n], *[cluster_vdos[m, :plot_n] for m in range(n_plot)]],
    ax=ax,
    xlim=(0, 200),
    ylim=(0, 0.6),
    colors=["black", *cluster_colors[:n_plot]],
    labels=["Total VDoS", *[f"Mode {m + 1}" for m in range(n_plot)]],
    linestyles=["-", *["--"] * n_plot],
    vlines=[freqs[i] for i in cluster_peak_idx[:n_plot]],
    vline_colors=cluster_colors[:n_plot],
    title="Clustered key vibrations",
)
plot_spectra(
    plot_freqs,
    np.sum(cluster_vdos[:n_plot, :plot_n], axis=0),
    ax=ax,
    colors="gray",
    labels="Sum of modes",
    legend=True,
)
plt.tight_layout()
plt.show()


In [ ]:
overlap = mode_overlap_matrix(cluster_modes)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(overlap, vmin=0, vmax=1, cmap="inferno")
ax.set_title("Clustered mode overlap")
ax.set_xlabel("mode")
ax.set_ylabel("mode")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()
